# The Super Parameter

**PennyLane Coding Challenge — QHack 2023 Flashback Badge · Intermediate · Quantum Machine Learning**

## Challenge statement

Within QML it is common to find the term *expressivity*, which refers to the size of all possible models that we can generate by varying our parameters. One way to increase the expressivity of our model family is usually by adding more parameters. However, this is not always a good thing, since increasing the number of parameters, and therefore the number of possible models, means that we have to perform our training on a very large set, making it more difficult to find the model that best suits our needs. Therefore, the real challenge of a good QML researcher is to find the smallest possible family of models that still contains the optimal solution. There is much more to the notion of expressivity, but in this challenge we are going to push the concept to its limits.

Suppose that we are in the situation where we have 3 qubits and we know that the solution to our problem is a computational basis state, i.e. an element of the set

$$\{|000\rangle, |001\rangle, |010\rangle, |011\rangle, |100\rangle, |101\rangle, |110\rangle, |111\rangle\}.$$

We don't know exactly what the basis state is, so we would like to generate an ansatz expressive enough so that

$$U(\vec\alpha)\,|000\rangle = |b_0 b_1 b_2\rangle$$

for certain values of $\vec\alpha$. An example of ansatz that accomplishes this would be a circuit with one $R_X(\alpha_i)$ on every qubit. This is the fundamental concept in **Basis embedding**, where you can see that by taking $\alpha_0$, $\alpha_1$ and $\alpha_2$ properly, we can generate any basis state. However, this challenge is not going to be this easy. You are asked to build an ansatz that, **with only one parameter**, is able to generate all the basis states. To judge your solution, we will ask you to provide us with a list of the 8 values of the parameter that generate each of them.

## Challenge code

You must complete the qnode `model` that will be in charge of obtaining different outputs. This model depends on a single parameter and you must ensure that it generates all the basis states. You must also define the function `generate_coefficients`, which will return a list with the 8 values of the parameter to generate these basis states.

## Output

To judge this challenge, the `generate_coefficients` function will be called first. With the output of this function (the eight coefficients), we will call the model to ensure that the generated states are the desired ones. In addition, we will check that:

- The model is continuous (small modifications of the parameter imply small modifications of the generated state). By putting the parameter inside rotation gates you will have no problems with this.
- The generated coefficients are in the interval $[0, 10]$. Solutions that do not fit this interval will be considered incorrect.

In this challenge, we will not work with public and private tests. We will simply check that all of the above is fulfilled.

## Approach: a binary counter in the $R_X$ basis

Let the parameter take the integer values $\alpha_n = n \in \{0,\dots,7\}$ and write $n = 4b_0 + 2b_1 + b_2$ (wire 0 is the most-significant bit, matching the ordering of `qml.probs`).

An $R_X(\theta)$ acting on $|0\rangle$ yields $|1\rangle$ with probability $\sin^2(\theta/2)$, so the state is a basis state exactly when $\theta$ is an integer multiple of $\pi$, and it is $|1\rangle$ when that integer is odd. The idea is to give every wire a net rotation angle of $\pi \times (\text{its own bit})$:

| wire | free rotation | controlled corrections | net angle at $\alpha = n$ |
|---|---|---|---|
| 2 (LSB) | $R_X(\pi\alpha)$ | — | $\pi n = \pi(b_2 + 2b_1 + 4b_0)$ → parity $b_2$ |
| 1 | $R_X(\pi\alpha/2)$ | $CR_X(-\pi/2)$ from wire 2 | $\pi(n - b_2)/2 = \pi(b_1 + 2b_0)$ → parity $b_1$ |
| 0 (MSB) | $R_X(\pi\alpha/4)$ | $CR_X(-\pi/4)$ from wire 2, $CR_X(-\pi/2)$ from wire 1 | $\pi(n - b_2 - 2b_1)/4 = \pi b_0$ |

Each wire's free rotation runs at half the frequency of the wire below it, and the controlled gates subtract exactly the contribution of the lower bits. Because the lower wires are already in a definite basis state when they act as controls, the corrections are applied deterministically, and every wire ends up in $|b_k\rangle$ up to a global phase.

**Why a single layer of uncontrolled rotations cannot work.** With $R_X(c_k\alpha)$ on each wire, every $c_k\alpha_n/\pi$ must be an integer with parity $b_k(n)$. The state $|001\rangle$ then forces $c_0/c_2 = \text{even}/\text{odd}$ while $|100\rangle$ forces $c_0/c_2 = \text{odd}/\text{even}$, which is impossible for a single rational number, so some entangling correction is unavoidable.

The coefficients are $[0, 1, 2, 3, 4, 5, 6, 7]$, all inside $[0, 10]$, and the parameter only ever appears inside rotation angles, so the model is continuous.

In [1]:
import json
import pennylane as qp
import pennylane.numpy as np

dev = qp.device("default.qubit", wires=3)

In [2]:
@qp.qnode(dev)
def model(alpha):
    """In this qnode you will define your model in such a way that there is a single 
    parameter alpha which returns each of the basic states.

    Args:
        alpha (float): The only parameter of the model.

    Returns:
        (numpy.tensor): The probability vector of the resulting quantum state.
    """

    # Put your code here #
    # Wire 2 (least-significant bit): one full flip per unit of alpha.
    qp.RX(np.pi * alpha, wires=2)

    # Wire 1: half the frequency, then remove the LSB's share of the angle.
    qp.RX(np.pi * alpha / 2, wires=1)
    qp.CRX(-np.pi / 2, wires=[2, 1])

    # Wire 0 (most-significant bit): quarter frequency, then remove the two lower bits' shares.
    qp.RX(np.pi * alpha / 4, wires=0)
    qp.CRX(-np.pi / 4, wires=[2, 0])
    qp.CRX(-np.pi / 2, wires=[1, 0])

    return qp.probs(wires=[0, 1, 2])


def generate_coefficients():
    """This function must return a list of 8 different values of the parameter that
    generate the states 000, 001, 010, ..., 111, respectively, with your ansatz.

    Returns:
        (list(int)): A list of eight real numbers.
    """
    return [0, 1, 2, 3, 4, 5, 6, 7]

### Circuit

In [3]:
print(qp.draw(model, decimals=2)(alpha=1.0))

0: ──RX(0.79)────────────╭RX(-0.79)─╭RX(-1.57)─┤ ╭Probs
1: ──RX(1.57)─╭RX(-1.57)─│──────────╰●─────────┤ ├Probs
2: ──RX(3.14)─╰●─────────╰●────────────────────┤ ╰Probs


### Quick check: one basis state per coefficient

In [4]:
coefs = generate_coefficients()
for i, c in enumerate(coefs):
    probs = model(c)
    print(f"alpha = {c}  ->  |{i:03b}>   p = {float(probs[i]):.6f}   (full vector: {np.round(probs, 6)})")

alpha = 0  ->  |000>   p = 1.000000   (full vector: [1. 0. 0. 0. 0. 0. 0. 0.])
alpha = 1  ->  |001>   p = 1.000000   (full vector: [0. 1. 0. 0. 0. 0. 0. 0.])
alpha = 2  ->  |010>   p = 1.000000   (full vector: [0. 0. 1. 0. 0. 0. 0. 0.])
alpha = 3  ->  |011>   p = 1.000000   (full vector: [0. 0. 0. 1. 0. 0. 0. 0.])
alpha = 4  ->  |100>   p = 1.000000   (full vector: [0. 0. 0. 0. 1. 0. 0. 0.])
alpha = 5  ->  |101>   p = 1.000000   (full vector: [0. 0. 0. 0. 0. 1. 0. 0.])
alpha = 6  ->  |110>   p = 1.000000   (full vector: [0. 0. 0. 0. 0. 0. 1. 0.])
alpha = 7  ->  |111>   p = 1.000000   (full vector: [0. 0. 0. 0. 0. 0. 0. 1.])


### Official test harness

This is the grading code from the challenge, unchanged. The continuity check evaluates the circuit at 10,000 points in $[0, 10)$ (three evaluations each), so this cell takes about a minute on `default.qubit`.

In [5]:
# These functions are responsible for testing the solution.
def run(test_case_input: str) -> str:
    return None

def check(solution_output, expected_output: str) -> None:
    coefs = generate_coefficients()
    output = np.array([model(c) for c in coefs])
    epsilon = 0.001

    for i in range(len(coefs)):
        assert np.isclose(output[i][i], 1)

    def is_continuous(function, point):
        limit = calculate_limit(function, point)

        if limit is not None and sum(abs(limit - function(point))) < epsilon:
            return True
        else:
            return False

    def is_continuous_in_interval(function, interval):
        for point in interval:
            if not is_continuous(function, point):
                return False
        return True

    def calculate_limit(function, point):
        x_values = [point - epsilon, point, point + epsilon]
        y_values = [function(x) for x in x_values]
        average = sum(y_values) / len(y_values)

        return average

    assert is_continuous_in_interval(model, np.arange(0,10,0.001))

    for coef in coefs:
        assert coef >= 0 and coef <= 10

# These are the public test cases
test_cases = [
    ('No input', 'No output')
]
# This will run the public test cases locally
for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)

    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        if message := check(output, expected_output):
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")

        else:
            print("Correct!")

Running test case 0 with input 'No input'...


Correct!
